In [1]:
import torch

print("="*50)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("CUDA Version :", torch.version.cuda)
    print("GPU Memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
else:
    print("GPU not found!")
print("="*50)

CUDA Available : True
GPU : Tesla T4
CUDA Version : 12.8
GPU Memory (GB): 14.56


In [2]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.7 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/oct_instruction.jsonl"
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 92
    })
})

In [4]:
dataset = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 73
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 19
    })
})

In [5]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer

In [6]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [8]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [9]:
model = AutoModelForCausalLM.from_pretrained(

    model_name,

    quantization_config=bnb_config,

    device_map="auto"

)

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [10]:
model = prepare_model_for_kbit_training(model)

In [11]:
peft_config = LoraConfig(

    r=16,

    lora_alpha=32,

    target_modules=[

        "q_proj",

        "k_proj",

        "v_proj",

        "o_proj"

    ],

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM"

)

In [12]:
model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [13]:
def formatting_func(example):

    return f"""### Instruction

{example['instruction']}

### Input

{example['input']}

### Response

{example['output']}
"""

In [18]:
training_args = TrainingArguments(

    output_dir="./qwen_oct",

    num_train_epochs=5,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=5,

    save_strategy="epoch",

    eval_strategy="epoch",

    fp16=False,

    report_to="none"

)

In [19]:
trainer = SFTTrainer(

    model=model,

    train_dataset=dataset["train"],

    eval_dataset=dataset["test"],

    formatting_func=formatting_func,

    args=training_args

)

In [20]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,2.525996,2.496295,2.647301,55667.000000,0.510660
2,2.313597,2.230174,2.271242,111334.000000,0.545211
3,2.033593,1.985671,2.122517,167001.000000,0.584549
4,1.862764,1.867482,1.961035,222668.000000,0.605333
5,1.744933,1.828614,1.935821,278335.000000,0.609853


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=50, training_loss=2.1574463844299316, metrics={'train_runtime': 180.4525, 'train_samples_per_second': 2.023, 'train_steps_per_second': 0.277, 'total_flos': 671320399411200.0, 'train_loss': 2.1574463844299316, 'epoch': 5.0})

In [21]:
trainer.model.save_pretrained(
    "/content/drive/MyDrive/qwen_oct_lora"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/qwen_oct_lora"
)

('/content/drive/MyDrive/qwen_oct_lora/tokenizer_config.json',
 '/content/drive/MyDrive/qwen_oct_lora/chat_template.jinja',
 '/content/drive/MyDrive/qwen_oct_lora/tokenizer.json')

In [22]:
from transformers import pipeline

pipe = pipeline(

    "text-generation",

    model=model,

    tokenizer=tokenizer

)

prompt = """
### Instruction

Classify the following OCT report.

### Input

RNFL Single Exam Report OU ...

### Response

"""

result = pipe(

    prompt,

    max_new_tokens=30

)

print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces 


### Instruction

Classify the following OCT report.

### Input

RNFL Single Exam Report OU ...

### Response

This is an R report. It should be classified as either:

OCT-1 (Organic Chemical Test)
OCT-2 (Physical Examination


In [23]:
!pip install fastapi uvicorn pyngrok